# NER Using Hidden Markov Model (HMM)
### Alexandria University – NLP Assignment 2 | Part 2

This notebook implements:
- A Hidden Markov Model (HMM) trained on the CoNLL-2003 train split
- Transition and emission probability estimation with Laplace smoothing
- Viterbi algorithm for sequence decoding
- Evaluation (accuracy, precision, recall, F1-score) on the test split

## 1. Install & Import Dependencies

In [ ]:
!pip install datasets seqeval --quiet

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('All imports successful.')

## 2. Load CoNLL-2003 Dataset

In [ ]:
dataset = load_dataset('lhoestq/conll2003', trust_remote_code=True)
print(dataset)

# NER tag mapping
label_names = dataset['train'].features['ner_tags'].feature.names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(f'\nNER labels: {label_names}')

## 3. Extract Train & Test Sentences

In [ ]:
def extract_sentences(split):
    """Return list of (tokens, ner_label_strings) tuples."""
    sentences = []
    for example in dataset[split]:
        tokens = example['tokens']
        tags   = [id2label[t] for t in example['ner_tags']]
        sentences.append((tokens, tags))
    return sentences

train_sentences = extract_sentences('train')
test_sentences  = extract_sentences('test')

print(f'Train sentences : {len(train_sentences)}')
print(f'Test  sentences : {len(test_sentences)}')
print(f'\nExample sentence:')
print('Tokens:', train_sentences[0][0])
print('Tags  :', train_sentences[0][1])

## 4. Preprocessing
Lowercasing tokens; keeping punctuation as-is (HMM uses surface forms).

In [ ]:
def preprocess_token(token):
    """Lowercase the token."""
    return token.lower()

def preprocess_sentences(sentences):
    return [([preprocess_token(t) for t in tokens], tags)
            for tokens, tags in sentences]

train_data = preprocess_sentences(train_sentences)
test_data  = preprocess_sentences(test_sentences)

print('Preprocessing done.')
print('Example after preprocessing:', train_data[0][0][:8])

## 5. HMM Training
### 5.1 Count-based estimation

We estimate:
- **Initial probabilities** π(tag) – probability of a sentence starting with `tag`
- **Transition probabilities** A(tag_i | tag_{i-1})
- **Emission probabilities** B(word | tag)

All probabilities are smoothed with **Laplace (add-1) smoothing** to handle unseen events.

In [ ]:
class HMM:
    """
    First-order Hidden Markov Model for NER.
    Probabilities are stored in log-space to avoid underflow.
    """

    def __init__(self, smoothing=1.0):
        self.smoothing   = smoothing      # Laplace smoothing factor
        self.tags        = []             # unique NER tags
        self.vocab       = set()          # training vocabulary
        self.log_pi      = {}             # log initial probs
        self.log_A       = {}             # log transition probs
        self.log_B       = {}             # log emission probs
        self.unk_token   = '<UNK>'

    # ------------------------------------------------------------------
    def fit(self, sentences):
        """Estimate HMM parameters from labelled sentences."""
        k = self.smoothing

        # ── raw count dictionaries ──────────────────────────────────────
        init_counts  = defaultdict(float)         # tag at position 0
        trans_counts = defaultdict(lambda: defaultdict(float))  # prev→cur
        emit_counts  = defaultdict(lambda: defaultdict(float))  # tag→word
        tag_counts   = defaultdict(float)         # total occurrences of tag

        all_tags = set()
        vocab    = set()

        for tokens, tags in sentences:
            if not tokens:
                continue
            init_counts[tags[0]] += 1
            for i, (word, tag) in enumerate(zip(tokens, tags)):
                emit_counts[tag][word] += 1
                tag_counts[tag]        += 1
                all_tags.add(tag)
                vocab.add(word)
                if i > 0:
                    trans_counts[tags[i-1]][tag] += 1

        self.tags  = sorted(all_tags)
        self.vocab = vocab
        n_tags = len(self.tags)
        vocab.add(self.unk_token)           # reserve UNK slot
        n_vocab = len(vocab)

        # ── initial probabilities (π) ───────────────────────────────────
        total_sents = sum(init_counts.values()) + k * n_tags
        self.log_pi = {
            tag: np.log((init_counts[tag] + k) / total_sents)
            for tag in self.tags
        }

        # ── transition probabilities (A) ────────────────────────────────
        self.log_A = {}
        for prev in self.tags:
            denom = sum(trans_counts[prev].values()) + k * n_tags
            self.log_A[prev] = {
                cur: np.log((trans_counts[prev][cur] + k) / denom)
                for cur in self.tags
            }

        # ── emission probabilities (B) ──────────────────────────────────
        self.log_B = {}
        for tag in self.tags:
            denom = tag_counts[tag] + k * n_vocab
            word_probs = {
                word: np.log((emit_counts[tag][word] + k) / denom)
                for word in vocab
            }
            # UNK = smoothed prob for an unseen word
            word_probs[self.unk_token] = np.log(k / denom)
            self.log_B[tag] = word_probs

        print(f'HMM trained on {len(sentences)} sentences.')
        print(f'Tags  : {self.tags}')
        print(f'Vocab : {len(self.vocab):,} words')

    # ------------------------------------------------------------------
    def _emit(self, tag, word):
        """Return log-emission prob; use UNK for OOV words."""
        if word in self.log_B[tag]:
            return self.log_B[tag][word]
        return self.log_B[tag][self.unk_token]

    # ------------------------------------------------------------------
    def viterbi(self, tokens):
        """
        Viterbi decoding.
        Returns the most-likely tag sequence for `tokens`.
        """
        n      = len(tokens)
        tags   = self.tags
        n_tags = len(tags)

        # viterbi[t][s] = best log-prob to reach state s at step t
        viterbi  = np.full((n, n_tags), -np.inf)
        backptr  = np.zeros((n, n_tags), dtype=int)

        # ── initialise ──────────────────────────────────────────────────
        word0 = tokens[0] if tokens[0] in self.vocab else self.unk_token
        for s, tag in enumerate(tags):
            viterbi[0, s] = self.log_pi[tag] + self._emit(tag, word0)

        # ── recursion ───────────────────────────────────────────────────
        for t in range(1, n):
            word = tokens[t] if tokens[t] in self.vocab else self.unk_token
            for s, tag in enumerate(tags):
                emit = self._emit(tag, word)
                scores = [
                    viterbi[t-1, p] + self.log_A[tags[p]][tag] + emit
                    for p in range(n_tags)
                ]
                best_prev       = int(np.argmax(scores))
                viterbi[t, s]   = scores[best_prev]
                backptr[t, s]   = best_prev

        # ── backtrack ───────────────────────────────────────────────────
        best_last = int(np.argmax(viterbi[n-1]))
        path = [best_last]
        for t in range(n-1, 0, -1):
            path.append(backptr[t, path[-1]])
        path.reverse()

        return [tags[s] for s in path]

    # ------------------------------------------------------------------
    def predict(self, sentences):
        """Run Viterbi on each sentence; return list of predicted tag sequences."""
        return [self.viterbi(tokens) for tokens, _ in sentences]


# ── Instantiate & train ────────────────────────────────────────────────
hmm = HMM(smoothing=1.0)
hmm.fit(train_data)

## 6. Visualise Learned Probabilities
### 6.1 Transition Probability Heatmap

In [ ]:
tags = hmm.tags

# Build transition matrix (convert from log to prob)
trans_matrix = np.exp(
    [[hmm.log_A[prev][cur] for cur in tags] for prev in tags]
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    trans_matrix,
    xticklabels=tags, yticklabels=tags,
    annot=True, fmt='.3f', cmap='Blues', ax=ax
)
ax.set_title('HMM Transition Probabilities  P(tag_i | tag_{i-1})', fontsize=13)
ax.set_xlabel('Current Tag')
ax.set_ylabel('Previous Tag')
plt.tight_layout()
plt.savefig('hmm_transition_heatmap.png', dpi=150)
plt.show()

### 6.2 Initial Probability Bar Chart

In [ ]:
pi_probs = [np.exp(hmm.log_pi[t]) for t in tags]

plt.figure(figsize=(10, 4))
bars = plt.bar(tags, pi_probs, color='steelblue', edgecolor='white')
plt.title('HMM Initial Probabilities  π(tag)', fontsize=13)
plt.ylabel('Probability')
plt.xlabel('NER Tag')
for bar, prob in zip(bars, pi_probs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{prob:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('hmm_initial_probs.png', dpi=150)
plt.show()

## 7. Viterbi Decoding on Test Set

In [ ]:
from tqdm.auto import tqdm

print('Running Viterbi on test set …')
y_pred_seqs = []
for tokens, _ in tqdm(test_data):
    y_pred_seqs.append(hmm.viterbi(tokens))

print(f'Decoded {len(y_pred_seqs)} sentences.')

## 8. Evaluation

In [ ]:
# Flatten token-level predictions and ground-truth
y_true_flat = [tag for _, tags in test_data for tag in tags]
y_pred_flat = [tag for seq in y_pred_seqs for tag in seq]

assert len(y_true_flat) == len(y_pred_flat), 'Length mismatch!'

# ── Token-level metrics ────────────────────────────────────────────────
acc   = accuracy_score(y_true_flat, y_pred_flat)
prec  = precision_score(y_true_flat, y_pred_flat, average='weighted', zero_division=0)
rec   = recall_score(y_true_flat, y_pred_flat, average='weighted', zero_division=0)
f1    = f1_score(y_true_flat, y_pred_flat, average='weighted', zero_division=0)

# Macro (treat each class equally – better for imbalanced NER labels)
macro_f1 = f1_score(y_true_flat, y_pred_flat, average='macro', zero_division=0)

print('='*50)
print('     HMM NER — Token-level Evaluation')
print('='*50)
print(f'  Accuracy          : {acc:.4f}')
print(f'  Precision (W-avg) : {prec:.4f}')
print(f'  Recall    (W-avg) : {rec:.4f}')
print(f'  F1-score  (W-avg) : {f1:.4f}')
print(f'  F1-score  (Macro) : {macro_f1:.4f}')
print('='*50)

In [ ]:
# Per-class report
print('\n--- Per-class Classification Report ---')
print(classification_report(y_true_flat, y_pred_flat, target_names=label_names, zero_division=0))

### 8.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true_flat, y_pred_flat, labels=label_names)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_title('Confusion Matrix — Raw Counts', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Normalised
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_title('Confusion Matrix — Row-normalised', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('hmm_confusion_matrix.png', dpi=150)
plt.show()

### 8.2 Per-class F1 Bar Chart

In [ ]:
per_class_f1 = f1_score(y_true_flat, y_pred_flat,
                        labels=label_names, average=None, zero_division=0)

plt.figure(figsize=(11, 5))
colors = ['#2196F3' if f >= 0.7 else '#FF9800' if f >= 0.4 else '#F44336'
          for f in per_class_f1]
bars = plt.bar(label_names, per_class_f1, color=colors, edgecolor='white')
plt.axhline(f1, color='navy', linestyle='--', linewidth=1.5,
            label=f'Weighted avg F1 = {f1:.3f}')
plt.ylim(0, 1.05)
plt.title('HMM NER — Per-class F1 Score', fontsize=13)
plt.xlabel('NER Tag')
plt.ylabel('F1 Score')
plt.legend()
for bar, val in zip(bars, per_class_f1):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('hmm_per_class_f1.png', dpi=150)
plt.show()

## 9. Example Predictions

In [ ]:
print(f'{'Token':<20} {'True Tag':<12} {'Predicted Tag':<12} Match')
print('-' * 58)

for i in range(5):  # show first 5 test sentences
    tokens, true_tags = test_data[i]
    pred_tags = y_pred_seqs[i]
    print(f'\n--- Sentence {i+1} ---')
    for tok, tru, pre in zip(tokens, true_tags, pred_tags):
        match = '✓' if tru == pre else '✗'
        print(f'{tok:<20} {tru:<12} {pre:<12} {match}')

## 10. Summary

| Metric | HMM |
|---|---|
| Accuracy | — |
| Precision (weighted) | — |
| Recall (weighted) | — |
| F1-score (weighted) | — |
| F1-score (macro) | — |

> Fill the table above with your actual results after running the notebook.

### Key observations
- The HMM relies entirely on surface-form statistics (transition + emission counts).
- Laplace smoothing handles OOV words by assigning a small non-zero probability.
- The `O` (outside) class dominates the data, inflating weighted-average metrics.
- Macro F1 is a better indicator of performance on minority entity classes (PER, LOC, ORG, MISC).
- Compared to the Feed-Forward Neural Network, the HMM is faster to train but typically has lower recall on rare entity types because it cannot generalise via distributed representations.